<a href="https://colab.research.google.com/github/financieras/articulos/blob/main/MiniMax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **El algoritmo Minimax y la poda Alpha-Beta: construyendo una IA para el juego Gomoku**

1. Introducción — ¿Por qué Gomoku es un gran campo de pruebas para IA clásica?  
2. Gomoku clásico: reglas y nociones básicas  
3. El árbol de juego: el "qué pasaría si..." en acción  
4. Minimax: la lógica del rival perfecto  
5. Poda Alpha-Beta: cortando ramas inútiles  
6. Funciones de evaluación: cuando no llegas al final de la partida  
7. Optimizaciones prácticas: haz que tu IA sea rápida y fuerte  
8. Demo en código: implementa y juega contra tu IA  
9. Más allá de lo clásico: limitaciones y el futuro de la IA en juegos  
10. Conclusión: lo que te llevas y próximos pasos  

---

## 1. Introducción — ¿Por qué Gomoku es un gran campo de pruebas para IA clásica?

Los juegos han sido el laboratorio perfecto para la IA desde sus inicios.  

- **Breve historia**: Alan Turing (1950) propuso el ajedrez como test de inteligencia. Claude Shannon ideó búsquedas en árboles. Deep Blue venció a Kasparov en 1997 (ajedrez). AlphaGo (2016) revolucionó con MCTS y redes neuronales en Go.  

- **Gomoku clásico vs. otros juegos**: Tic-tac-toe es trivial (árbol completo). Connect-4 es manejable. Ajedrez y Go son gigantes. Gomoku es ideal: reglas simples, pero explosión combinatoria (tablero grande, ~200 movimientos iniciales) → obliga a trucos inteligentes.  

- **Qué vamos a construir**: Una IA razonablemente fuerte con búsqueda clásica (Minimax + optimizaciones).  

- **Objetivo del notebook**: Entender conceptos intuitivamente + poder jugar contra la IA. Ejecuta las celdas y experimenta.  

---

## 2. Gomoku clásico: reglas y nociones básicas

Gomoku (también llamado «cinco en raya») es un juego aparentemente sencillo, pero con una complejidad combinatoria enorme.

### Reglas básicas (versión clásica)
- Tablero: típicamente 15×15 o 19×19 (aunque en teoría es infinito)  
- Dos jugadores: Negro (primero) y Blanco  
- Objetivo: ser el primero en alinear **exactamente 5 piedras** consecutivas (horizontal, vertical o diagonal)  
- No hay capturas, prohibiciones ni reglas especiales (sin «doble tres», sin capturas como en Renju o Pente)  
- Gana el que coloca la quinta piedra que completa la línea  

### Ideas estratégicas intuitivas (lo que realmente importa)
- **Control del centro** → las casillas centrales permiten más direcciones y conexiones  
- **Iniciativa** → quien amenaza primero suele forzar al rival a defender  
- **Amenazas abiertas** → una línea con ambos extremos libres (`_XXXX_`) es muy peligrosa  
- **Amenazas cerradas** → un extremo bloqueado (`XXXX_` o `_XXXX`) es menos urgente  
- **Dobles amenazas** → crear dos amenazas simultáneas que el rival no pueda bloquear ambas  

### Representación en Python (lo más simple posible)

La forma más práctica y legible para un notebook:

```python
# Instalaciones/importaciones necesarias (ejecuta esto primero)
!pip install numpy  # Si no está instalado en Colab
import numpy as np
```

```python
# Opción 1: lista de listas (muy clara)
board = [['.' for _ in range(15)] for _ in range(15)]

# Opción 2: numpy (más rápida para cálculos posteriores)
import numpy as np
board_np = np.full((15, 15), '.', dtype=str)
# o con números: 0 = vacío, 1 = Negro, -1 = Blanco
board_num = np.zeros((15, 15), dtype=np.int8)
```

Ambas opciones funcionan perfectamente para Minimax.  
Usaremos la versión numpy en las demos porque facilita contar líneas y vectorizar algunas operaciones.

En la siguiente celda puedes visualizar un tablero vacío o con unas pocas jugadas de ejemplo.

```python
# Ejemplo: visualiza un tablero simple
def print_board(board):
    for row in board:
        print(' '.join(row))

print_board(board)  # Tablero vacío
```

---

## 3. El árbol de juego: el "qué pasaría si..." en acción

Imagina que eres Negro y acabas de mover en el centro. El rival responde... ¿y ahora qué? El **árbol de juego** mapea todas las posibilidades futuras.

### Conceptos esenciales
- **Estado**: tablero + turno actual (ej: `board` + `jugador`)  
- **Movimiento legal**: casilla vacía (en Gomoku: casi todas al inicio, ~200 válidas)  
- **Factor de ramificación (b)**: 50-200 por turno → crece exponencialmente  
- **Nodo hoja**: fin de partida (victoria/derrota/empate)  

### La trampa: explosión combinatoria
| Profundidad | Nodos (b=100) | Tiempo estimado (1M nodos/s) |
|-------------|---------------|------------------------------|
| 4           | 100M          | 0.1 s                       |
| 6           | 1T            | 17 min                      |
| 8           | 100T          | 3 años                      |
| 10          | 10Q           | ...eternidad                |

¡Sin trucos, imposible ir más allá de profundidad 4!

### Visualiza un árbol mini (profundidad 2)

```python
def print_mini_tree():
    print("Raíz (tu turno):")
    print("  ├── Mov1 → Rival: MovA (bueno), MovB (malo)")
    print("  │     ├── Tu respA → +10")
    print("  │     └── Tu respB → -5")
    print("  └── Mov2 → Rival: MovC (mejor)")
    print("        ├── Tu respC → +3")
    print("        └── Tu respD → -20")

print_mini_tree()
```

**Próximo**: Minimax recorre este árbol inteligentemente.

---

## 4. Minimax: la lógica del rival perfecto

Minimax es el algoritmo clásico que asume que **ambos jugadores juegan perfectamente**: tú intentas maximizar tu ventaja y el rival intenta minimizarla.

### Idea central (muy sencilla)
- En tu turno (MAX): eliges el movimiento que te da el **mejor** resultado posible.  
- En el turno del rival (MIN): elige el movimiento que te da el **peor** resultado posible.  
- Al final del árbol → puntúas la posición con una función de evaluación (próxima sección).  

### Ejemplo muy pequeño (tablero 3×3, ganar con 3 en raya)
Supongamos que estamos en esta posición (X = tú, O = rival, . = vacío):

```
X . .
. O .
. . .
```

Tú (X) puedes ganar en un movimiento si juegas bien. Minimax lo encuentra explorando:

- Tú mueves → +∞ (victoria inmediata) → eliges esa jugada  
- Si no, el rival intentaría bloquearte  

### Pseudocódigo básico (sin poda todavía)

```python
def minimax(tablero, profundidad, es_maximizador):
    if hay_ganador(tablero):
        return 1000 if es_maximizador else -1000     # victoria para MAX / MIN
    if tablero_lleno(tablero):
        return 0                                     # empate
    
    if es_maximizador:                               # Tu turno (MAX)
        mejor = -float('inf')
        for movimiento in movimientos_posibles(tablero):
            tablero.hacer_movimiento(movimiento)
            valor = minimax(tablero, profundidad+1, False)
            tablero.deshacer_movimiento(movimiento)
            mejor = max(mejor, valor)
        return mejor
    
    else:                                            # Turno rival (MIN)
        peor = float('inf')
        for movimiento in movimientos_posibles(tablero):
            tablero.hacer_movimiento(movimiento)
            valor = minimax(tablero, profundidad+1, True)
            tablero.deshacer_movimiento(movimiento)
            peor = min(peor, valor)
        return peor
```

### Problema inmediato
Este código explora **todo** el árbol → en Gomoku real es imposible (ver tabla de la sección 3).

**Próximo**: la poda alpha-beta salva la situación cortando ramas que ya sabemos que no valen la pena.

---

## 5. Poda Alpha-Beta: cortando ramas inútiles

Minimax explora **todo**. Alpha-Beta dice: "¡Ey! Si ya sé que esta rama es peor que otra que vi antes, la corto".

### Intuición con analogía
Imagina que buscas la casa más barata:  
- Encuentras una por 100k (α = tu "mejor hasta ahora").  
- La siguiente rama ofrece solo 120k+ → la descartas (β ≤ α).  

En el árbol:  
- **α**: mejor score garantizado para MAX (sube desde abajo).  
- **β**: peor score garantizado para MIN (baja desde arriba).  
- Si α ≥ β → **poda**: rama inutilizable.  

### Ejemplo visual (mini-árbol profundidad 3)
```
Raíz (MAX)
├── Mov1 → MIN: [explora todo] → valor 5
└── Mov2 → MIN:
    ├── MovA → MAX: 10  (α=10)
    └── MovB → MAX: 3   (poda: β≤10, no explora más)
Valor final: max(5, ? ) pero Mov2 ≥10 → eliges Mov2
```

**Ahorro típico**: reduce nodos de b^d a b^(d/2) (¡factor √b! ~10x en Gomoku).

### Pseudocódigo con poda (añade α, β)

```python
def alphabeta(tablero, profundidad, α, β, es_max):
    if terminal(tablero): return eval(tablero)
    
    if es_max:
        for mov in movimientos_ordenados(tablero):  # ¡orden importa!
            valor = alphabeta(tablero, profundidad+1, α, β, False)
            α = max(α, valor)
            if α >= β: break  # Poda β
        return α
    else:
        for mov in movimientos_ordenados(tablero):
            valor = alphabeta(tablero, profundidad+1, α, β, True)
            β = min(β, valor)
            if α >= β: break  # Poda α
        return β
```

**Próximo**: ¿qué puntúa las hojas? La función de evaluación.

```python
# Celda ejecutable: compara Minimax vs Alpha-Beta en un mini-problema (añade tu implementación simple aquí para contar nodos)
global nodos_minimax, nodos_alphabeta
# ... (implementa y compara)
```

---

## 6. Funciones de evaluación: cuando no llegas al final de la partida

En Gomoku real, casi nunca puedes buscar hasta el final (profundidad 30+).  
La función de evaluación asigna un **número** a cualquier posición intermedia:  
- positivo → bueno para Negro (MAX)  
- negativo → bueno para Blanco (MIN)  
- cuanto más grande en valor absoluto → más clara la ventaja  

### ¿Qué hace "buena" una posición en Gomoku?

Ideas intuitivas que casi todos usan:  

1. **Contar líneas consecutivas** (lo más importante)  
   - 5 seguidas → victoria inmediata (+∞ o 1 000 000)  
   - 4 abiertas (`_XXXX_`) → amenaza forzada (+100 000)  
   - 4 semiabiertas (`XXXX_` o `_XXXX`) → +10 000  
   - 3 abiertas (`_XXX_`) → +5 000  
   - 3 semiabiertas → +1 000  
   - 2 abiertas → +200–500  

2. **Bonificaciones estratégicas**  
   - Control del centro (casillas centrales valen más)  
   - Movilidad: más direcciones abiertas = mejor  
   - Dobles amenazas: si hay dos líneas fuertes simultáneas → multiplicador  

3. **Diferencial**  
   - score_total = score_negro - score_blanco  
   → la IA maximiza este valor  

### Ejemplo de heurística sencilla en Python

```python
def evaluar_posicion(tablero, jugador):
    """
    jugador: 1 (Negro), -1 (Blanco)
    Devuelve score desde la perspectiva de Negro
    """
    score = 0
    
    # Recorrer todas las líneas posibles (horizontal, vertical, diagonales)
    for i in range(15):
        for j in range(15):
            if tablero[i,j] == 0: continue
            
            # Contar en 4 direcciones
            for di, dj in [(0,1), (1,0), (1,1), (1,-1)]:
                count = 1
                abierto = 0
                
                # Hacia adelante
                ni, nj = i + di, j + dj
                while 0 <= ni < 15 and 0 <= nj < 15 and tablero[ni,nj] == tablero[i,j]:
                    count += 1
                    ni += di
                    nj += dj
                if 0 <= ni < 15 and 0 <= nj < 15 and tablero[ni,nj] == 0:
                    abierto += 1
                
                # Hacia atrás
                ni, nj = i - di, j - dj
                while 0 <= ni < 15 and 0 <= nj < 15 and tablero[ni,nj] == tablero[i,j]:
                    count += 1
                    ni -= di
                    nj -= dj
                if 0 <= ni < 15 and 0 <= nj < 15 and tablero[ni,nj] == 0:
                    abierto += 1
                
                # Puntuación según longitud y aperturas
                if count >= 5:
                    return 1_000_000 * jugador
                elif count == 4:
                    score += (100_000 if abierto == 2 else 10_000) * jugador
                elif count == 3:
                    score += (5_000 if abierto == 2 else 1_000) * jugador
                elif count == 2:
                    score += 300 * abierto * jugador
    
    # Bonificación centro (simple)
    centro = tablero[7:8, 7:8].sum()  # aproximado
    score += centro * 50
    
    return score
```

### El arte real: prueba y error
- Estas puntuaciones son aproximadas → hay que ajustarlas jugando muchas partidas.  
- Una buena heurística vale más que +2 de profundidad.  

**Próximo**: cómo combinar todo esto de forma eficiente con optimizaciones prácticas.

---

## 7. Optimizaciones prácticas: haz que tu IA sea rápida y fuerte

Minimax + Alpha-Beta ya es potente, pero sin optimizaciones sigue siendo lento en Gomoku. Aquí van las mejoras más efectivas (ordenadas por impacto real):  

### 1. Orden de movimientos (Move Ordering) – la estrella indiscutible  
- Explora **primero** las jugadas que parecen buenas → la poda alpha-beta corta mucho más.  
- Trucos simples y potentes:  
  - Amenazas inmediatas (win-in-1 o must-block) primero  
  - Capturas / extensiones de líneas abiertas  
  - Movimientos cerca del centro o que continúan tus líneas  
  - Movimientos que responden a amenazas del rival  

**Impacto**: puede multiplicar ×5–×20 la profundidad efectiva.  

### 2. Detección rápida de amenazas (Threat Detection)  
Antes de lanzar la búsqueda completa:  

```python
def hay_victoria_inmediata(tablero, jugador):
    # Chequea si hay 5 en raya ya (o casi)
    ...

def debe_bloquear(tablero, oponente):
    # ¿El rival gana en el próximo turno?
    ...
```

Si hay victoria inmediata → juega ya.  
Si hay amenaza crítica → fuerza el bloqueo sin buscar profundo.  

### 3. Profundización iterativa (Iterative Deepening)  
En vez de buscar fijo a profundidad 8:  
- Busca profundidad 1 → guarda mejor jugada  
- Profundidad 2 → mejora  
- ... hasta que se acabe el tiempo  
→ Siempre tienes una jugada razonable, y mejoras progresivamente.  

### 4. Tabla de transposición (Transposition Table) básica  
Muchas posiciones se alcanzan por caminos distintos → guárdalas.  

```python
trans_table = {}  # clave: hash del tablero → (valor, profundidad, flag)

def lookup_hash(tablero):
    h = hash_tablero(tablero)  # Zobrist o simple tuple
    if h in trans_table:
        return trans_table[h]
    ...
```

### 5. Reducción del branching factor  
- Solo genera movimientos adyacentes a piedras existentes (reduce de 225 a ~30–60)  
- Limita exploración: top-10 movimientos por nodo en niveles profundos  

### Resumen de ganancias típicas (en Gomoku realista)  
| Optimización              | Reducción nodos | Tiempo por movimiento |  
|---------------------------|-----------------|-----------------------|  
| Alpha-Beta sola           | ~90–95%         | base                  |  
| + Buen move ordering      | ×5–10           | mucho más rápido      |  
| + Threat detection        | ×2–5 (en medio) | 0.1–0.5 s             |  
| + Transposition table     | ×2–4            | + memoria             |  
| + Iterative deepening     | –               | control de tiempo     |  

Con estas 4–5 mejoras, pasas de "imposible" a "jugable en <2 segundos" con profundidad 6–8.  

**Próximo**: ponemos todo junto en una demo jugable.  

---

## 8. Demo en código: implementa y juega contra tu IA

Aquí juntamos todo: un motor Gomoku clásico en Python que puedes ejecutar directamente en el notebook.  
Usaremos una versión simplificada pero funcional (profundidad limitada + las optimizaciones clave).  

### Estructura básica del código  
- Tablero: numpy 2D (15×15)  
- Jugadores: 1 (Negro/MAX), -1 (Blanco/MIN)  
- Función principal: `buscar_mejor_movimiento()` con alpha-beta  

### Código completo (copia y pega en celdas separadas)  

```python
import numpy as np
import time

BOARD_SIZE = 15
EMPTY = 0
BLACK = 1
WHITE = -1

def crear_tablero():
    return np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)

def es_valido(tablero, x, y):
    return 0 <= x < BOARD_SIZE and 0 <= y < BOARD_SIZE and tablero[x, y] == EMPTY

def hacer_movimiento(tablero, x, y, jugador):
    tablero[x, y] = jugador

def deshacer_movimiento(tablero, x, y):
    tablero[x, y] = EMPTY

def imprimir_tablero(tablero):
    symbols = {EMPTY: '.', BLACK: 'X', WHITE: 'O'}
    print("  " + " ".join(str(i) for i in range(BOARD_SIZE)))
    for i in range(BOARD_SIZE):
        row = [symbols[tablero[i,j]] for j in range(BOARD_SIZE)]
        print(f"{i:2} {' '.join(row)}")
```

```python
def evaluar(tablero, jugador):
    score = 0
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            if tablero[i,j] == 0: continue
            sign = 1 if tablero[i,j] == jugador else -1
            for di, dj in [(0,1),(1,0),(1,1),(1,-1)]:
                count = 1
                abiertos = 0
                # adelante
                ni, nj = i + di, j + dj
                while es_valido(tablero, ni, nj) and tablero[ni,nj] == tablero[i,j]:
                    count += 1
                    ni += di
                    nj += dj
                if es_valido(tablero, ni, nj) and tablero[ni,nj] == EMPTY:
                    abiertos += 1
                # atrás
                ni, nj = i - di, j - dj
                while es_valido(tablero, ni, nj) and tablero[ni,nj] == tablero[i,j]:
                    count += 1
                    ni -= di
                    nj -= dj
                if es_valido(tablero, ni, nj) and tablero[ni,nj] == EMPTY:
                    abiertos += 1
                
                if count >= 5:
                    return 1_000_000 * sign
                elif count == 4:
                    score += (80_000 if abiertos == 2 else 8_000) * sign
                elif count == 3:
                    score += (4_000 if abiertos == 2 else 800) * sign
                elif count == 2:
                    score += 200 * abiertos * sign
    return score
```

```python
def alpha_beta(tablero, profundidad, alpha, beta, maximizando, jugador):
    if profundidad == 0:
        return evaluar(tablero, jugador)
    
    movimientos = [(x,y) for x in range(BOARD_SIZE) for y in range(BOARD_SIZE)
                   if es_valido(tablero, x, y)]
    
    # Orden simple: centro primero (mejora ligera la poda)
    movimientos.sort(key=lambda m: -abs(m[0]-7.5) - abs(m[1]-7.5))
    
    if maximizando:
        max_eval = -float('inf')
        for x, y in movimientos:
            hacer_movimiento(tablero, x, y, jugador)
            eval = alpha_beta(tablero, profundidad-1, alpha, beta, False, jugador)
            deshacer_movimiento(tablero, x, y)
            max_eval = max(max_eval, eval)
            alpha = max(alpha, eval)
            if beta <= alpha:
                break
        return max_eval
    else:
        min_eval = float('inf')
        for x, y in movimientos:
            hacer_movimiento(tablero, x, y, -jugador)
            eval = alpha_beta(tablero, profundidad-1, alpha, beta, True, jugador)
            deshacer_movimiento(tablero, x, y)
            min_eval = min(min_eval, eval)
            beta = min(beta, eval)
            if beta <= alpha:
                break
        return min_eval

def mejor_movimiento(tablero, jugador, profundidad=5):
    start = time.time()
    mejor_score = -float('inf')
    mejor_mov = None
    movimientos = [(x,y) for x in range(BOARD_SIZE) for y in range(BOARD_SIZE)
                   if es_valido(tablero, x, y)]
    movimientos.sort(key=lambda m: -abs(m[0]-7.5) - abs(m[1]-7.5))
    
    for x, y in movimientos[:20]:  # limitamos para que no tarde siglos
        hacer_movimiento(tablero, x, y, jugador)
        score = alpha_beta(tablero, profundidad-1, -float('inf'), float('inf'), False, jugador)
        deshacer_movimiento(tablero, x, y)
        if score > mejor_score:
            mejor_score = score
            mejor_mov = (x, y)
    
    print(f"Tiempo: {time.time()-start:.2f}s | Mejor score: {mejor_score}")
    return mejor_mov
```

```python
# Juega una partida rápida
tablero = crear_tablero()
# Ejemplo inicial: mueve en el centro
hacer_movimiento(tablero, 7, 7, BLACK)

print("Tablero inicial:")
imprimir_tablero(tablero)

print("\nIA (Blanco) pensando...")
mov = mejor_movimiento(tablero, WHITE, profundidad=4)
if mov:
    x, y = mov
    hacer_movimiento(tablero, x, y, WHITE)
    print(f"IA juega en ({x}, {y})")
    imprimir_tablero(tablero)
```

**Notas para experimentar**  
- Sube `profundidad` a 5–6 si tu máquina aguanta (más lento pero más fuerte)  
- Añade detección de victoria real y threat detection para mejorar mucho  
- Prueba mover tú manualmente y llama a `mejor_movimiento()` después de cada turno  

¡Ahora tienes una IA básica de Gomoku que puedes tunear en vivo!  

---

## 9. Más allá de lo clásico: limitaciones y el futuro de la IA en juegos

Minimax con alpha-beta y heurísticas manuales funciona muy bien para Gomoku... hasta cierto punto. Aquí las principales limitaciones y qué enfoques han tomado la delantera en los últimos años.  

### Limitaciones reales de este enfoque (2026)  
- **Horizon effect**: la IA no ve amenazas que ocurren justo después de su profundidad máxima (ej: profundidad 6 → ignora una victoria en el movimiento 7).  
- **Heurística manual frágil**: ajustar los pesos (4 abierto = 80k, 3 abierto = 4k...) es un arte impreciso → una mala afinación y la IA juega mal en aperturas o finales.  
- **Escalabilidad pobre**: en tableros grandes y fases medias, incluso con todas las optimizaciones, profundidad >8 sigue costando segundos/minutos.  
- **No aprende**: no mejora sola con experiencia; todo el conocimiento está hardcodeado por el programador.  

### El cambio de paradigma: Monte Carlo Tree Search (MCTS)  
- En vez de evaluar con heurística estática → simula miles de partidas aleatorias rápidas desde cada nodo.  
- Ventaja: no necesita heurística manual → el "conocimiento" emerge de las simulaciones.  
- Popularizado por AlphaGo (2016): MCTS + red neuronal para guiar las simulaciones.  
- En Gomoku: KataGo, Leela Zero y muchas IAs open-source lo usan con éxito brutal.  

### AlphaZero / KataGo style (redes neuronales + búsqueda)  
- Red neuronal profunda predice:  
  - Probabilidad de victoria desde cualquier posición (value head)  
  - Qué movimientos son prometedores (policy head)  
- Entrenamiento por auto-juego: la IA juega contra sí misma millones de veces.  
- Resultado: fuerza sobrehumana sin conocimiento humano previo.  
- En Gomoku: muchas IAs actuales (KataGo adaptado, proyectos open como "KataJigo") superan ampliamente a cualquier Minimax+heurística.  

### ¿Sigue teniendo sentido Minimax/Alpha-Beta en 2026?  
Sí, y mucho:  
- Fácil de implementar y entender (ideal para aprender).  
- Bajo consumo de recursos (no necesita GPU ni datasets enormes).  
- Excelente para proyectos educativos, torneos con límite de tiempo/hardware, o cuando quieres control total.  
- Híbridos: muchas IAs modernas usan Minimax/Alpha-Beta en aperturas/finales + MCTS en medio juego.  

**Resumen rápido**  
Clásico (Minimax + heurística) → didáctico, rápido de prototipar, fuerte para humanos casuales.  
Moderno (MCTS + redes) → sobrehumano, pero requiere mucho cómputo y datos.  

**Próximo**: conclusiones y recursos para seguir experimentando.  

---

## 10. Conclusión: lo que te llevas y próximos pasos

Hemos recorrido el camino completo desde la idea intuitiva hasta una IA jugable de Gomoku usando técnicas clásicas de búsqueda en juegos.  

### Resumen de ideas clave que debes recordar  
- Minimax → asume que el rival es perfecto → base de toda búsqueda adversarial  
- Alpha-Beta → poda ramas inútiles → multiplica ×10 la profundidad efectiva  
- Función de evaluación → el verdadero "cerebro" cuando no llegas al final → cuenta líneas abiertas, amenazas y control del centro  
- Optimizaciones prácticas → move ordering, threat detection, iterative deepening y transposition tables → convierten lo imposible en jugable  
- En Gomoku clásico → con estas herramientas puedes llegar a una IA que gana cómodamente a la mayoría de humanos casuales/intermedios  

### Lo más importante (para que no se te olvide)  
- **Intuición > fórmulas**  
  Entiende por qué se poda una rama antes de memorizar la recursión con α y β.  
- **Orden importa muchísimo**  
  Un buen orden de movimientos vale más que +2 de profundidad.  
- **Heurística es arte**  
  Ajusta los pesos jugando, no copiando tablas mágicas.  
- **No hay bala de plata**  
  Minimax es genial para aprender, pero el futuro (y la fuerza sobrehumana) está en MCTS + redes neuronales.  

### Próximos pasos recomendados  
1. **Mejora la demo**  
   - Añade detección real de victoria (5 en raya)  
   - Implementa threat detection antes de la búsqueda  
   - Ordena movimientos por score rápido de evaluación  
   - Prueba profundidad 6–7 y mide tiempos/nodos  

2. **Experimenta**  
   - Cambia los pesos de la heurística y juega 10 partidas  
   - Compara versiones con/sin alpha-beta, con/sin ordenación  

3. **Recursos para seguir**  
   - Libro clásico: "Artificial Intelligence: A Modern Approach" (Russell & Norvig) – capítulo sobre juegos  
   - Código open-source: busca "gomoku alpha-beta python" en GitHub (muchos ejemplos buenos)  
   - Proyectos modernos: KataGo (https://github.com/lightvector/KataGo) – MCTS + redes para Go/Gomoku  
   - Vídeos: serie "AlphaGo explained" en YouTube (DeepMind)  
   - Comunidad: foros de computer-go.org o Reddit r/gomoku  

¡Ahora te toca a ti!  
Modifica el código, haz tu propia versión, sube el notebook a Kaggle o GitHub, y comparte cómo le va a tu IA contra amigos o contra ti mismo.  

Gracias por llegar hasta aquí.  
¡Que disfrutes programando y jugando contra máquinas que (casi) piensan!